In [0]:
import json, requests
from datetime import datetime, timezone

FEED = "https://cdn.mbta.com/realtime/Alerts_enhanced.json"
BASE = "/Volumes/transit/bronze/landing/rt/alerts"

resp = requests.get(FEED, timeout=30)
resp.raise_for_status()
raw = resp.content

header_ts = json.loads(raw)["header"]["timestamp"]
dt = datetime.fromtimestamp(header_ts, tz=timezone.utc).strftime("%Y-%m-%d")

out_dir = f"{BASE}/dt={dt}"
dbutils.fs.mkdirs(out_dir)
out_path = f"{out_dir}/snapshot_{header_ts}.json"

with open(out_path, "wb") as f:
    f.write(raw)          # verbatim bytes, no parsing

print(f"{out_path}  {len(raw):,} bytes  {len(json.loads(raw)['entity'])} alerts")

In [0]:
from datetime import datetime, timezone
base = "/Volumes/transit/bronze/landing/rt/alerts"
days = sorted(d.name.rstrip("/") for d in dbutils.fs.ls(base))
for d in days[-3:]:
    files = dbutils.fs.ls(f"{base}/{d}")
    newest = max(f.modificationTime for f in files) / 1000
    print(f"{d}  {len(files):>3} files  newest {(datetime.now(timezone.utc).timestamp()-newest)/60:.0f} min ago")